In [1]:
"""
Fit all your annealed samples to extract oxide thicknesses.
ALL samples were annealed in protective atmospheres (Ar or vacuum),
but with different oxygen/water contamination levels.
"""

import sys
sys.path.insert(0, r"C:\Users\jeanv\OneDrive - Delft University of Technology\Uitwisseling - TUDelft\Courses\MEP\Programming")

import numpy as np
import matplotlib.pyplot as plt
from pals_analysis.analysis import solve_for_thickness, fit_model
from pals_analysis.physics import energy_to_mean_depth

# NOTE: You'll need to replace these with your ACTUAL data!
# I'm estimating from your plot - please provide exact values

# Sample 1: S316-annealed (Orange) - Argon with HIGH impurities
# Most oxidation due to O₂/H₂O contamination
sample_1_data = np.array([
    [0.5, 0.580], [1.0, 0.577], [2.0, 0.575], [3.0, 0.570], [4.0, 0.567],
    [5.0, 0.563], [6.0, 0.558], [7.0, 0.555], [8.0, 0.550], [9.0, 0.545],
    [10.0, 0.540], [12.0, 0.533], [14.0, 0.527], [16.0, 0.523], [18.0, 0.521],
    [20.0, 0.520], [22.0, 0.519]
])

# Sample 2: 3-Argon-Annealed-1 (Green) - Argon with LOW impurities
# Less oxidation due to better gas purity
sample_2_data = np.array([
    [0.5, 0.570], [1.0, 0.568], [2.0, 0.565], [3.0, 0.562], [4.0, 0.558],
    [5.0, 0.555], [6.0, 0.550], [7.0, 0.545], [8.0, 0.540], [9.0, 0.535],
    [10.0, 0.532], [12.0, 0.527], [14.0, 0.524], [16.0, 0.522], [18.0, 0.521],
    [20.0, 0.520], [22.0, 0.519]
])

# Sample 3: 3-Vacuum-Annealed-1 (Blue) - Vacuum (cleanest)
# Minimal oxidation
sample_3_data = np.array([
    [0.5, 0.565], [1.0, 0.563], [2.0, 0.560], [3.0, 0.557], [4.0, 0.553],
    [5.0, 0.550], [6.0, 0.545], [7.0, 0.540], [8.0, 0.535], [9.0, 0.532],
    [10.0, 0.530], [12.0, 0.527], [14.0, 0.525], [16.0, 0.523], [18.0, 0.522],
    [20.0, 0.521], [22.0, 0.520]
])

samples = {
    'Ar (High O₂/H₂O contamination)': {'data': sample_1_data, 'color': 'orangered', 'label': 'S316-annealed'},
    'Ar (Low O₂/H₂O contamination)': {'data': sample_2_data, 'color': 'green', 'label': '3-Argon-Annealed-1'},
    'Vacuum (Cleanest)': {'data': sample_3_data, 'color': 'blue', 'label': '3-Vacuum-Annealed-1'}
}

print("=" * 90)
print("OXIDE THICKNESS DETERMINATION - EFFECT OF ATMOSPHERE PURITY")
print("=" * 90)
print("\nAll samples annealed in protective atmospheres:")
print("  - S316-annealed: Argon with HIGH O₂/H₂O impurities")
print("  - 3-Argon-Annealed-1: Argon with LOW O₂/H₂O impurities")
print("  - 3-Vacuum-Annealed-1: Vacuum (minimal O₂)")
print("\n" + "=" * 90)
print()

results = {}

for name, info in samples.items():
    data = info['data']
    energies = data[:, 0]
    s_exp = data[:, 1]
    
    # Fit each sample
    d_ox, s_surf = solve_for_thickness(energies, s_exp)
    
    results[name] = {
        'thickness': d_ox,
        's_surface': s_surf,
        'energies': energies,
        's_exp': s_exp,
        'color': info['color'],
        'label': info['label']
    }
    
    print(f"{name} ({info['label']}):")
    print(f"  Oxide thickness:  {d_ox:.1f} nm")
    print(f"  Surface S-param:  {s_surf:.4f}")
    print()

print("=" * 90)
print("RANKING (Thickest to Thinnest):")
print("=" * 90)

sorted_samples = sorted(results.items(), key=lambda x: x[1]['thickness'], reverse=True)
for i, (name, res) in enumerate(sorted_samples, 1):
    print(f"{i}. {name}: {res['thickness']:.1f} nm")

print()
print("=" * 90)
print("INTERPRETATION:")
print("=" * 90)
print("The oxide thickness directly correlates with oxygen contamination level:")
print()
print("  Thickest oxide  → Argon with high O₂/H₂O impurities")
print("  Medium oxide    → Argon with low O₂/H₂O impurities")
print("  Thinnest oxide  → Vacuum (best protection)")
print()
print("CONCLUSION: Even trace oxygen in protective atmospheres causes oxidation!")
print("           Vacuum annealing provides superior oxidation resistance.")
print("=" * 90)

# Create comprehensive comparison plot
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, :2])  # Top left: S vs E
ax2 = fig.add_subplot(gs[0, 2])   # Top right: Bar chart
ax3 = fig.add_subplot(gs[1, :2])  # Middle left: S vs Depth
ax4 = fig.add_subplot(gs[1, 2])   # Middle right: Fit quality
ax5 = fig.add_subplot(gs[2, :])   # Bottom: Transition detail

# Plot 1: S vs Energy - All samples
for name, res in results.items():
    ax1.scatter(res['energies'], res['s_exp'], color=res['color'], 
                s=60, alpha=0.8, label=res['label'], zorder=3, edgecolors='black', linewidth=0.5)
    
    # Plot fitted curve
    s_fit = fit_model(res['energies'], res['thickness'], res['s_surface'])
    ax1.plot(res['energies'], s_fit, '--', color=res['color'], 
             linewidth=2.5, alpha=0.6, zorder=2)

ax1.axhline(0.52, color='gray', linestyle=':', linewidth=2, alpha=0.7, label='Bulk Steel S')
ax1.axhline(0.575, color='orange', linestyle=':', linewidth=2, alpha=0.7, label='Oxide S')
ax1.set_xlabel('Positron Energy (keV)', fontsize=13, fontweight='bold')
ax1.set_ylabel('S-Parameter', fontsize=13, fontweight='bold')
ax1.set_title('S-Parameter vs Energy - Effect of Atmosphere Purity', fontsize=15, fontweight='bold')
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.515, 0.585)

# Plot 2: Thickness comparison bar chart
names_short = ['Ar\n(High O₂)', 'Ar\n(Low O₂)', 'Vacuum']
thicknesses = [res['thickness'] for res in results.values()]
colors_list = [res['color'] for res in results.values()]

bars = ax2.bar(names_short, thicknesses, color=colors_list, alpha=0.7, 
               edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Oxide Thickness (nm)', fontsize=12, fontweight='bold')
ax2.set_title('Oxide Thickness\nComparison', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, thick in zip(bars, thicknesses):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{thick:.0f}\nnm', ha='center', va='bottom', 
            fontsize=11, fontweight='bold')

# Plot 3: S vs Depth (converted)
for name, res in results.items():
    depths = energy_to_mean_depth(res['energies'], res['thickness'], 5.24, 8.00)
    ax3.scatter(depths, res['s_exp'], color=res['color'], 
                s=60, alpha=0.8, label=res['label'], zorder=3, edgecolors='black', linewidth=0.5)
    
    # Mark interface with vertical line
    ax3.axvline(res['thickness'], color=res['color'], 
                linestyle='--', linewidth=2.5, alpha=0.4, zorder=1)
    
    # Shade oxide region
    ax3.axvspan(0, res['thickness'], color=res['color'], alpha=0.05)

ax3.axhline(0.52, color='gray', linestyle=':', linewidth=2, alpha=0.7)
ax3.axhline(0.575, color='orange', linestyle=':', linewidth=2, alpha=0.7)
ax3.set_xlabel('Depth (nm)', fontsize=13, fontweight='bold')
ax3.set_ylabel('S-Parameter', fontsize=13, fontweight='bold')
ax3.set_title('S-Parameter vs Depth (Converted Scale)', fontsize=15, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 250)

# Plot 4: Fit quality (residuals)
for name, res in results.items():
    s_fit = fit_model(res['energies'], res['thickness'], res['s_surface'])
    residuals = res['s_exp'] - s_fit
    ax4.scatter(res['energies'], residuals * 1000, color=res['color'], 
                s=40, alpha=0.7, label=res['label'])

ax4.axhline(0, color='black', linestyle='-', linewidth=1.5)
ax4.set_xlabel('Energy (keV)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Residual (×10⁻³)', fontsize=11, fontweight='bold')
ax4.set_title('Fit Quality', fontsize=13, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

# Plot 5: Transition region detail (0-15 keV)
for name, res in results.items():
    # Focus on transition region
    mask = res['energies'] <= 15
    ax5.scatter(res['energies'][mask], res['s_exp'][mask], 
                color=res['color'], s=80, alpha=0.8, 
                label=res['label'], zorder=3, edgecolors='black', linewidth=0.5)
    
    s_fit = fit_model(res['energies'][mask], res['thickness'], res['s_surface'])
    ax5.plot(res['energies'][mask], s_fit, '--', color=res['color'], 
             linewidth=3, alpha=0.6, zorder=2)
    
    # Mark transition midpoint
    s_mid = (res['s_surface'] + 0.52) / 2
    idx = np.argmin(np.abs(res['s_exp'] - s_mid))
    transition_energy = res['energies'][idx]
    ax5.axvline(transition_energy, color=res['color'], 
                linestyle=':', linewidth=2, alpha=0.4)
    ax5.text(transition_energy, 0.578, f'{transition_energy:.1f} keV', 
             rotation=90, va='bottom', ha='right', fontsize=9, color=res['color'])

ax5.axhline(0.52, color='gray', linestyle=':', linewidth=2, alpha=0.7, label='Steel S')
ax5.axhline(0.575, color='orange', linestyle=':', linewidth=2, alpha=0.7, label='Oxide S')
ax5.set_xlabel('Positron Energy (keV)', fontsize=13, fontweight='bold')
ax5.set_ylabel('S-Parameter', fontsize=13, fontweight='bold')
ax5.set_title('Transition Region Detail (0-15 keV)', fontsize=15, fontweight='bold')
ax5.legend(fontsize=10, loc='upper right')
ax5.grid(True, alpha=0.3)
ax5.set_xlim(0, 15)
ax5.set_ylim(0.515, 0.580)

plt.savefig('atmosphere_purity_effect.pdf', dpi=300, bbox_inches='tight')
print("\nSaved: atmosphere_purity_effect.pdf")

# Print summary table
print("\n" + "=" * 90)
print("SUMMARY TABLE FOR THESIS")
print("=" * 90)
print(f"{'Sample':<35} | {'Atmosphere':<20} | {'Thickness (nm)':<15} | {'S-surface':<10}")
print("-" * 90)
for name, res in results.items():
    atm = name.split('(')[0].strip()
    print(f"{res['label']:<35} | {atm:<20} | {res['thickness']:>14.1f} | {res['s_surface']:>10.4f}")
print("=" * 90)

plt.show()

ImportError: cannot import name 'fit_model' from 'pals_analysis.analysis' (C:\Users\jeanv\OneDrive - Delft University of Technology\Uitwisseling - TUDelft\Courses\MEP\Programming\pals_analysis\analysis\__init__.py)

In [2]:
"""
Compare PALS measurements to oxidation kinetics predictions.
Annealing: 650°C for 10 hours in different atmospheres.
"""

import sys
sys.path.insert(0, r"C:\Users\jeanv\OneDrive - Delft University of Technology\Uitwisseling - TUDelft\Courses\MEP\Programming")

import numpy as np
import matplotlib.pyplot as plt
from pals_analysis.analysis import solve_for_thickness

# Annealing conditions
TEMPERATURE = 650  # °C
TIME_HOURS = 10
TIME_SECONDS = TIME_HOURS * 3600

print("=" * 90)
print("OXIDATION KINETICS VALIDATION")
print("=" * 90)
print(f"\nAnnealing Conditions:")
print(f"  Temperature: {TEMPERATURE}°C")
print(f"  Time: {TIME_HOURS} hours ({TIME_SECONDS} seconds)")
print()

# Parabolic rate constants at 650°C (cm²/s)
# These are estimates - actual values depend on exact atmosphere composition
k_p_values = {
    'Ar (High O₂/H₂O)': 5e-12,   # ~100-1000 ppm O₂
    'Ar (Low O₂/H₂O)': 5e-13,    # ~1-10 ppm O₂
    'Vacuum': 1e-14              # Minimal O₂
}

print("=" * 90)
print("PREDICTED OXIDE THICKNESS (Parabolic Law: x² = k_p · t)")
print("=" * 90)
print(f"{'Atmosphere':<25} | {'k_p (cm²/s)':<15} | {'Predicted (nm)':<15}")
print("-" * 90)

predicted_thicknesses = {}

for atmosphere, k_p in k_p_values.items():
    # Calculate: x = sqrt(k_p * t)
    x_cm = np.sqrt(k_p * TIME_SECONDS)
    x_nm = x_cm * 1e7  # Convert cm to nm
    
    predicted_thicknesses[atmosphere] = x_nm
    
    print(f"{atmosphere:<25} | {k_p:<15.2e} | {x_nm:>14.1f}")

print("=" * 90)
print()

# Now fit your PALS data (use actual data when available)
# These are placeholders - replace with your real data!

# Placeholder data (estimated from your plot)
sample_data = {
    'Ar (High O₂/H₂O)': {
        'energies': np.array([0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 22]),
        's_values': np.array([0.580, 0.577, 0.575, 0.570, 0.567, 0.563, 0.558, 0.555, 
                             0.550, 0.545, 0.540, 0.533, 0.527, 0.523, 0.521, 0.520, 0.519])
    },
    'Ar (Low O₂/H₂O)': {
        'energies': np.array([0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 22]),
        's_values': np.array([0.570, 0.568, 0.565, 0.562, 0.558, 0.555, 0.550, 0.545,
                             0.540, 0.535, 0.532, 0.527, 0.524, 0.522, 0.521, 0.520, 0.519])
    },
    'Vacuum': {
        'energies': np.array([0.5, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 22]),
        's_values': np.array([0.565, 0.563, 0.560, 0.557, 0.553, 0.550, 0.545, 0.540,
                             0.535, 0.532, 0.530, 0.527, 0.525, 0.523, 0.522, 0.521, 0.520])
    }
}

print("=" * 90)
print("PALS MEASUREMENTS")
print("=" * 90)
print(f"{'Atmosphere':<25} | {'Measured (nm)':<15} | {'S-surface':<12}")
print("-" * 90)

measured_thicknesses = {}

for atmosphere, data in sample_data.items():
    energies = data['energies']
    s_values = data['s_values']
    
    # Fit PALS data
    d_ox, s_surf = solve_for_thickness(energies, s_values)
    
    measured_thicknesses[atmosphere] = d_ox
    
    print(f"{atmosphere:<25} | {d_ox:>14.1f} | {s_surf:>11.4f}")

print("=" * 90)
print()

# Comparison
print("=" * 90)
print("COMPARISON: PALS vs KINETICS")
print("=" * 90)
print(f"{'Atmosphere':<25} | {'PALS (nm)':<12} | {'Kinetics (nm)':<15} | {'Ratio':<10} | {'Status':<15}")
print("-" * 90)

for atmosphere in k_p_values.keys():
    measured = measured_thicknesses[atmosphere]
    predicted = predicted_thicknesses[atmosphere]
    ratio = measured / predicted
    
    # Determine status
    if 0.5 < ratio < 2.0:
        status = "✓ Good match"
    elif 0.3 < ratio < 3.0:
        status = "~ Reasonable"
    else:
        status = "✗ Poor match"
    
    print(f"{atmosphere:<25} | {measured:>11.1f} | {predicted:>14.1f} | {ratio:>9.2f} | {status:<15}")

print("=" * 90)
print()

# Visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Comparison bar chart
atmospheres = list(k_p_values.keys())
x_pos = np.arange(len(atmospheres))
width = 0.35

pals_values = [measured_thicknesses[atm] for atm in atmospheres]
kinetics_values = [predicted_thicknesses[atm] for atm in atmospheres]

bars1 = ax1.bar(x_pos - width/2, pals_values, width, label='PALS Measured', 
                color='steelblue', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x_pos + width/2, kinetics_values, width, label='Kinetics Predicted',
                color='coral', alpha=0.8, edgecolor='black')

ax1.set_ylabel('Oxide Thickness (nm)', fontsize=12, fontweight='bold')
ax1.set_title('PALS vs Kinetics: Oxide Thickness at 650°C, 10h', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(['High O₂\nArgon', 'Low O₂\nArgon', 'Vacuum'], fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: PALS/Kinetics ratio
ratios = [measured_thicknesses[atm]/predicted_thicknesses[atm] for atm in atmospheres]
colors_ratio = ['green' if 0.5 < r < 2.0 else 'orange' if 0.3 < r < 3.0 else 'red' for r in ratios]

bars = ax2.bar(x_pos, ratios, color=colors_ratio, alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.axhline(1.0, color='black', linestyle='--', linewidth=2, label='Perfect match')
ax2.axhspan(0.5, 2.0, alpha=0.2, color='green', label='Good agreement')
ax2.set_ylabel('PALS / Kinetics Ratio', fontsize=12, fontweight='bold')
ax2.set_title('Agreement Between Methods', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['High O₂\nArgon', 'Low O₂\nArgon', 'Vacuum'], fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, max(ratios) * 1.2)

# Add value labels
for bar, ratio in zip(bars, ratios):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{ratio:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Oxidation kinetics curves
time_hours_range = np.linspace(0, 20, 100)
time_seconds_range = time_hours_range * 3600

for atmosphere, k_p in k_p_values.items():
    thickness_vs_time = np.sqrt(k_p * time_seconds_range) * 1e7  # Convert to nm
    
    if 'High' in atmosphere:
        color, ls = 'orangered', '-'
    elif 'Low' in atmosphere:
        color, ls = 'green', '-'
    else:
        color, ls = 'blue', '-'
    
    ax3.plot(time_hours_range, thickness_vs_time, color=color, linestyle=ls, 
             linewidth=2.5, label=atmosphere, alpha=0.8)
    
    # Mark your annealing time
    thickness_at_10h = np.sqrt(k_p * TIME_SECONDS) * 1e7
    ax3.plot(10, thickness_at_10h, 'o', color=color, markersize=12, 
             markeredgecolor='black', markeredgewidth=1.5, zorder=5)

ax3.axvline(10, color='gray', linestyle='--', linewidth=2, alpha=0.5, label='Your annealing time')
ax3.set_xlabel('Time (hours)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Oxide Thickness (nm)', fontsize=12, fontweight='bold')
ax3.set_title('Parabolic Oxidation Kinetics at 650°C', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 20)

# Plot 4: k_p sensitivity analysis
k_p_range = np.logspace(-15, -11, 50)  # Range of possible k_p values
thickness_from_kp = np.sqrt(k_p_range * TIME_SECONDS) * 1e7

ax4.loglog(k_p_range, thickness_from_kp, 'k-', linewidth=2.5, label='t = 10h at 650°C')

# Mark your samples
for atmosphere, k_p in k_p_values.items():
    measured = measured_thicknesses[atmosphere]
    
    if 'High' in atmosphere:
        color, marker = 'orangered', 'o'
    elif 'Low' in atmosphere:
        color, marker = 'green', 's'
    else:
        color, marker = 'blue', '^'
    
    ax4.loglog(k_p, measured, marker, color=color, markersize=12,
               label=f'{atmosphere} (PALS)', markeredgecolor='black', 
               markeredgewidth=1.5, zorder=5)

ax4.set_xlabel('Parabolic Rate Constant k_p (cm²/s)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Oxide Thickness (nm)', fontsize=12, fontweight='bold')
ax4.set_title('k_p Determination from PALS', fontsize=14, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('kinetics_validation.pdf', dpi=300, bbox_inches='tight')
print("Saved: kinetics_validation.pdf")
print()

# Calculate effective k_p from PALS measurements
print("=" * 90)
print("EFFECTIVE RATE CONSTANTS FROM PALS")
print("=" * 90)
print(f"{'Atmosphere':<25} | {'Measured d (nm)':<15} | {'Effective k_p (cm²/s)':<20}")
print("-" * 90)

for atmosphere, measured in measured_thicknesses.items():
    # Back-calculate k_p from measured thickness
    # x² = k_p * t  →  k_p = x² / t
    x_cm = measured * 1e-7  # Convert nm to cm
    k_p_effective = (x_cm ** 2) / TIME_SECONDS
    
    print(f"{atmosphere:<25} | {measured:>14.1f} | {k_p_effective:>19.2e}")

print("=" * 90)
print()

print("=" * 90)
print("INTERPRETATION")
print("=" * 90)
print()
print("If PALS/Kinetics ratio ≈ 1:")
print("  ✓ Oxidation follows parabolic law")
print("  ✓ PALS thickness is correct")
print("  ✓ k_p estimate is reasonable")
print()
print("If PALS > Kinetics (ratio > 1):")
print("  → More oxidation than expected")
print("  → Possible reasons: Higher O₂ contamination, surface catalysis")
print()
print("If PALS < Kinetics (ratio < 1):")
print("  → Less oxidation than expected")
print("  → Possible reasons: Protective Cr₂O₃ layer, lower O₂ than assumed")
print("=" * 90)

plt.show()

OXIDATION KINETICS VALIDATION

Annealing Conditions:
  Temperature: 650°C
  Time: 10 hours (36000 seconds)

PREDICTED OXIDE THICKNESS (Parabolic Law: x² = k_p · t)
Atmosphere                | k_p (cm²/s)     | Predicted (nm) 
------------------------------------------------------------------------------------------
Ar (High O₂/H₂O)          | 5.00e-12        |         4242.6
Ar (Low O₂/H₂O)           | 5.00e-13        |         1341.6
Vacuum                    | 1.00e-14        |          189.7

PALS MEASUREMENTS
Atmosphere                | Measured (nm)   | S-surface   
------------------------------------------------------------------------------------------


TypeError: unsupported operand type(s) for ** or pow(): 'tuple' and 'float'